# Lab 4: An LLM Extraction You Can Trust

---

## What this lab is

In Lab 3 you built emissions numbers from plant-level data. Many numbers a climate-finance team needs only exist as **text** in sustainability reports, so analysts increasingly use an LLM to turn report text into a table.

This lab treats the LLM as a **measurement instrument**, exactly like the utilisation rates and emission factors of Lab 3: it has an error rate, the error rate has an **interval**, some errors are random and some are **systematic**, and a risk committee needs to know which is which.

* **Part A. From text to table.** Define a schema, parse LLM output, and label a few excerpts yourself.
* **Part B. Measure the instrument.** Accuracy per field, an interval on accuracy, and what errors do to a sector total.
* **Part C. Self-consistency.** Ask the model 5 times. Does disagreement tell you where it is wrong?
* **Part D. Retrieval.** A mini-RAG: can we even find the right paragraph?
* **Part E. Communicate.** Should the bank adopt this pipeline?

## Learning goals

1. Turn free text into **validated structured data** (JSON + pydantic).
2. Evaluate an LLM against a **gold standard** and put an **interval** on its accuracy.
3. Distinguish **random** LLM errors (caught by resampling) from **systematic** ones (not caught).
4. Design a **human-review policy** from an uncertainty signal and a cost.
5. Explain why **retrieval** is a separate source of error in RAG.

## Conventions

* Questions are **[QUAL]** (2-4 sentences) or **[QUANT]** (a number from your code plus one sentence).
* Write each answer in the cell right under the question.
* Units: emissions in **tCO2e** (tonnes), production in **Mt** (million tonnes) of crude steel.
* Boxes marked **[OPTIONAL]** can be skipped.
* Run **Kernel -> Restart & Run All** before submitting.

## About the data

All 20 companies are **fictional**, but the excerpts copy the way real reports are written (different units, fiscal years, combined scopes, missing values). `recorded_responses.json` contains LLM answers **recorded in advance** (1 answer at temperature 0 and 5 answers at temperature 0.7 per excerpt) so that everyone works on the same outputs and no API key is needed. If you have access to an LLM, the optional box in A2 lets you run your own and compare.

## Submission

Work in **groups of up to 4**. Fill in every member before submitting.

1. Full name: Lara Normand  Student ID: B00820883
2. Full name: Alix Bauret  Student ID: B00822419
3. Full name: Camille Metz  Student ID: B00821603
4. Full name: Amine Benbachir Hassani  Student ID: B00821087
5. Full name: Tiphaine De Saint Aignan  Student ID: B00819186

**Group / repo name:** `aidams-lab4-bauret-metz-normand-benbachir-desaintaignan`
**Submitter (one person):** Lara Normand
**Repo URL:** https://github.com/Lara-Normand-Pro/aidams-lab4-bauret-metz-normand-benbachir-desaintaignan

### What to submit

- This notebook (`lab4.ipynb`) with all tasks completed and cells run (outputs visible)
- A short `README.md` with how to run the notebook

Create a GitHub repo, push the notebook and README, invite the instructor, and paste the **repo URL** above.

**Send submission info to my email (1 email per group).** Include the GitHub repo URL.

**Done when:** all names are filled in and the notebook contains outputs (no need for the instructor to re-run it).

## Setup (given)

In [1]:
import json, re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pydantic import BaseModel, ValidationError

pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

DATA = Path("data")
excerpts = pd.DataFrame(json.loads((DATA / "excerpts.json").read_text()))
gold = (pd.DataFrame(json.loads((DATA / "gold_labels.json").read_text())).set_index("id")
          .astype({"report_year": "Int64", "net_zero_year": "Int64"}))
recorded = pd.DataFrame(json.loads((DATA / "recorded_responses.json").read_text()))
corpus = json.loads((DATA / "aurelia_report_chunks.json").read_text())

FIELDS = ["company", "report_year", "scope1_tco2", "scope2_tco2", "production_mt", "net_zero_year"]
SEED = 42
print(excerpts.shape, gold.shape, recorded.shape, recorded.kind.value_counts().to_dict())

(20, 2) (20, 6) (120, 6) {'sample': 100, 'greedy': 20}


### The prompt and the LLM connection (given)

`ask_llm(prompt, temperature)` returns the model's raw text. With `BACKEND = "recorded"` (default) you do not need it: the answers are already in `recorded`. The other backends are there for the optional live run.

In [2]:
PROMPT = """You extract climate data from a steel company's report excerpt.
Return ONLY a JSON object with these keys:
  company        (string)
  report_year    (integer, the year the figures refer to; for a fiscal year use the calendar year it starts in)
  scope1_tco2    (number in tonnes CO2e, or null if not reported separately)
  scope2_tco2    (number in tonnes CO2e, market-based if both are given, or null if not reported separately)
  production_mt  (crude steel production in million metric tonnes, or null)
  net_zero_year  (integer year of a net zero / carbon neutrality target, or null if there is none)
  confidence     (number between 0 and 1: how confident you are that every field is correct)
Convert all units. Never compute a value that is not reported.

Excerpt:
{text}
"""

BACKEND = "recorded"          # "recorded" | "anthropic" | "openai" | "ollama"
MODEL = None                  # e.g. "claude-haiku-5-5" (anthropic), your course model (openai), "llama3.2" (ollama)


def ask_llm(prompt: str, temperature: float = 0.0) -> str:
    if BACKEND == "anthropic":
        import anthropic                                   # pip install anthropic; needs ANTHROPIC_API_KEY
        msg = anthropic.Anthropic().messages.create(
            model=MODEL, max_tokens=400, temperature=temperature,
            messages=[{"role": "user", "content": prompt}])
        return msg.content[0].text
    if BACKEND == "openai":
        from openai import OpenAI                          # pip install openai; needs OPENAI_API_KEY
        r = OpenAI().chat.completions.create(
            model=MODEL, temperature=temperature,
            messages=[{"role": "user", "content": prompt}])
        return r.choices[0].message.content
    if BACKEND == "ollama":
        import requests                                    # needs a local `ollama serve`
        r = requests.post("http://localhost:11434/api/generate", timeout=120, json={
            "model": MODEL, "prompt": prompt, "stream": False, "options": {"temperature": temperature}})
        return r.json()["response"]
    raise RuntimeError("BACKEND is 'recorded': use the `recorded` table instead of calling the model.")


print(PROMPT.format(text=excerpts.text[0]))

You extract climate data from a steel company's report excerpt.
Return ONLY a JSON object with these keys:
  company        (string)
  report_year    (integer, the year the figures refer to; for a fiscal year use the calendar year it starts in)
  scope1_tco2    (number in tonnes CO2e, or null if not reported separately)
  scope2_tco2    (number in tonnes CO2e, market-based if both are given, or null if not reported separately)
  production_mt  (crude steel production in million metric tonnes, or null)
  net_zero_year  (integer year of a net zero / carbon neutrality target, or null if there is none)
  confidence     (number between 0 and 1: how confident you are that every field is correct)
Convert all units. Never compute a value that is not reported.

Excerpt:
Aurelia Steel - Annual Climate Report 2024. In 2024 Aurelia Steel produced 12.6 million tonnes of crude steel across its three integrated sites. Gross Scope 1 emissions were 24,190,000 tCO2e, and Scope 2 emissions (market-based)

---
# PART A. From text to table

## A0. Be the annotator first

> **Pause & think.** No code. Do this **before** you look at `gold`.

### Q1 [QUAL] Label three excerpts yourself

Print the excerpts **E03**, **E09** and **E19** and fill in the six fields for each by hand, following the rules in `PROMPT`. Then compare with `gold.loc[["E03", "E09", "E19"]]`. Which field was hardest for you, and why? If two people in your group disagreed, what does that say about the "truth" we will score the LLM against?

In [3]:
ids = ["E03", "E09", "E19"]
for i in ids:
    print(i, "->", excerpts.set_index("id").text[i], "\n")

# My hand labels, following the rules in PROMPT (no peeking at gold).
# E19: 3.3 million SHORT tons -> metric: 3.3 * 0.90718 = 2.99 Mt
mine = pd.DataFrame([
    dict(id="E03", company="Lianhe Iron & Steel Group", report_year=2023,
         scope1_tco2=4108 * 1e4, scope2_tco2=512 * 1e4, production_mt=21.4, net_zero_year=2060),
    dict(id="E09", company="Ostend Metals", report_year=2024,
         scope1_tco2=np.nan, scope2_tco2=np.nan, production_mt=4.4, net_zero_year=2050),
    dict(id="E19", company="Bluewater Iron", report_year=2024,
         scope1_tco2=5.1e6, scope2_tco2=410_000, production_mt=round(3.3 * 0.90718, 2), net_zero_year=2050),
]).set_index("id").astype({"report_year": "Int64", "net_zero_year": "Int64"})

print("MY LABELS"); display(mine)
print("GOLD");      display(gold.loc[ids])

# field-by-field agreement between me and gold (both-missing counts as agreement)
agree = {}
for c in mine.columns:
    if c == "company":
        agree[c] = [True] * 3
    else:
        a, b = mine[c].astype(float), gold.loc[ids, c].astype(float)
        agree[c] = (np.isclose(a, b, rtol=0.02) | (a.isna() & b.isna())).tolist()
print("Agreement with gold:"); display(pd.DataFrame(agree, index=ids))

E03 -> Lianhe Iron & Steel Group - Environmental, Social and Governance Report 2023. In 2023 the Group produced 21.4 million tonnes of crude steel. Greenhouse gas emissions (unit: 10,000 tonnes CO2): direct emissions (Scope 1) 4,108; indirect emissions from purchased electricity (Scope 2) 512. The Group will strive to peak carbon emissions before 2030 and achieve carbon neutrality by 2060. 

E09 -> Ostend Metals NV - Sustainability Statement 2024 (ESRS E1). Crude steel production: 4.4 Mt. Combined Scope 1 and Scope 2 emissions amounted to 8.7 MtCO2e, a 4% decrease year on year. Ostend Metals is committed to net zero emissions by 2050. 

E19 -> Bluewater Iron Corp. - 2024 Environmental Report. Shipments and production: we produced 3.3 million short tons of raw steel at our DRI-EAF facilities. Scope 1 emissions: 5.1 million metric tonnes CO2e; Scope 2: 410,000 metric tonnes CO2e. Bluewater targets net zero by 2050. 

MY LABELS


,company,report_year,scope1_tco2,scope2_tco2,production_mt,net_zero_year
id,,,,,,
E03,Lianhe Iron & Steel Group,2023,"41,080,000.000","5,120,000.000",21.400,2060
E09,Ostend Metals,2024,NaN,NaN,4.400,2050
E19,Bluewater Iron,2024,"5,100,000.000","410,000.000",2.990,2050


GOLD


,company,report_year,scope1_tco2,scope2_tco2,production_mt,net_zero_year
id,,,,,,
E03,Lianhe Iron & Steel Group,2023,"41,080,000.000","5,120,000.000",21.400,2060
E09,Ostend Metals,2024,NaN,NaN,4.400,2050
E19,Bluewater Iron,2024,"5,100,000.000","410,000.000",2.990,2050


Agreement with gold:


,company,report_year,scope1_tco2,scope2_tco2,production_mt,net_zero_year
E03,True,True,True,True,True,True
E09,True,True,True,True,True,True
E19,True,True,True,True,True,True


**Your answer (Q1):**

My labels matched `gold` on all 18 fields, but three traps made it slower than it looks. **E03**: the table header says "unit: 10,000 tonnes CO2", so 4,108 means 41.08 MtCO2 and the scale is easy to miss (this was the hardest one). **E19**: production is in *short* tons, so 3.3 becomes 2.99 Mt metric (3.3 x 0.907). **E09**: only a *combined* Scope 1+2 figure is given, and the prompt says not to compute anything, so both scopes are null.

If two people disagree, it shows that the "truth" is a set of conventions chosen by one annotator (null vs split for E09, rounding 2.99 vs 3.0, which year for a fiscal year), not an observation. The gold labels have their own error rate, so "accuracy" really means agreement with this annotation guide. Ideally two annotators label independently and we report their agreement first.

## A1. A schema and a parser

**Goal.** Create

* a pydantic model `SteelDisclosure` with the six fields of the prompt plus `confidence` (types: `str`, `int`, `Optional[float]`, `Optional[int]` as appropriate);
* a function `parse_response(raw)` that finds the JSON object inside the raw text, validates it with `SteelDisclosure`, and returns `(record, None)` on success or `(None, error_message)` on failure;
* a DataFrame `pred` with one row per **successfully parsed** response (columns `id`, `kind`, `sample` and the fields) and a list `failures`.

> **Hint.** Models often wrap JSON in text or code fences: `re.search(r"\{.*\}", raw, flags=re.S)`. Then `SteelDisclosure.model_validate_json(...)` and `record.model_dump()`. Catch `ValidationError`.

In [4]:
class SteelDisclosure(BaseModel):
    company: str
    report_year: int
    scope1_tco2: Optional[float] = None
    scope2_tco2: Optional[float] = None
    production_mt: Optional[float] = None
    net_zero_year: Optional[int] = None
    confidence: float


def parse_response(raw: str):
    """Find the JSON object in raw text, validate it. Returns (record, None) or (None, error_message)."""
    m = re.search(r"\{.*\}", raw, flags=re.S)
    if m is None:
        return None, "no JSON object found"
    try:
        record = SteelDisclosure.model_validate_json(m.group(0))
    except ValidationError as e:
        return None, "; ".join(f"{'.'.join(map(str, err['loc']))}: {err['msg']}" for err in e.errors())
    return record, None


rows, failures = [], []
for r in recorded.itertuples():
    rec, err = parse_response(r.raw_text)
    if rec is None:
        failures.append({"id": r.id, "kind": r.kind, "sample": r.sample, "error": err, "raw_text": r.raw_text})
    else:
        rows.append({"id": r.id, "kind": r.kind, "sample": r.sample, **rec.model_dump()})

pred = pd.DataFrame(rows)
print(f"parsed: {len(pred)} / {len(recorded)}   failures: {len(failures)}")
pred.head()

parsed: 119 / 120   failures: 1


,id,kind,sample,company,report_year,scope1_tco2,scope2_tco2,production_mt,net_zero_year,confidence
0,E01,greedy,0,Aurelia Steel,2024,"24,190,000.000","1,310,000.000",12.600,"2,050.000",0.910
1,E01,sample,0,Aurelia Steel,2024,"24,190,000.000","1,310,000.000",12.600,"2,050.000",0.940
2,E01,sample,1,Aurelia Steel,2024,"24,190,000.000","1,310,000.000",12.600,"2,050.000",0.910
3,E01,sample,2,Aurelia Steel,2024,"24,190,000.000","1,310,000.000",12.600,"2,050.000",0.930
4,E01,sample,3,Aurelia Steel,2024,"24,190,000.000","1,310,000.000",12.600,"2,050.000",0.960


### Q2 [QUANT + QUAL] What breaks?

(a) How many of the 120 recorded responses fail to parse, and why (look at their `raw_text`)? How many responses needed the regex because they were wrapped in text? (b) Give **two** options for handling a failed response in a production pipeline, and one risk of each. In Part B we will count a failed response as **wrong on every field**: why is that the honest choice for an accuracy figure?

In [5]:
n_fail = len(failures)
print("Failed to parse:", n_fail)
for f in failures:
    print(f["id"], f["kind"], f["sample"], "->", f["error"])
    print("   raw_text:", repr(f["raw_text"][:200]))

# needed the regex = plain json.loads on the raw text does NOT work, but parse_response does
def plain_json_ok(raw):
    try:
        json.loads(raw); return True
    except Exception:
        return False

wrapped = recorded[~recorded.raw_text.map(plain_json_ok)]
fail_keys = {(f["id"], f["kind"], f["sample"]) for f in failures}
n_regex = sum((r.id, r.kind, r["sample"]) not in fail_keys for _, r in wrapped.iterrows())
print(f"\nResponses wrapped in text / code fences (regex needed and it worked): {n_regex}")
print(f"Responses that parsed cleanly as plain JSON: {len(recorded) - len(wrapped)}")

Failed to parse: 1
E11 greedy 0 -> production_mt: Input should be a valid number, unable to parse string as a number
   raw_text: '{\n  "company": "Anatolia Celik",\n  "report_year": 2024,\n  "scope1_tco2": 4120000,\n  "scope2_tco2": 2350000,\n  "production_mt": "8.1 Mt",\n  "net_zero_year": 2053,\n  "confidence": 0.94\n}'

Responses wrapped in text / code fences (regex needed and it worked): 7
Responses that parsed cleanly as plain JSON: 113


**Your answer (Q2):**

**(a)** 1 of the 120 responses fails to parse: the greedy answer for **E11**, where `production_mt` is the *string* "8.1 Mt" instead of a number (it also picked shipments, 8.1, instead of production, 9.8). 7 responses were wrapped in text ("Sure! Here is the extracted data" plus a code fence) and needed the regex; the regex rescued all of them. The other 113 were plain JSON.

**(b)** Option 1: retry or re-prompt with the validation error. Risk: cost and latency, and the retry can return a well-formed but wrong value, so a format fix is mistaken for a correctness fix. Option 2: send it to a human queue. Risk: it is slow and expensive, and failures cluster on the hardest documents, so the queue can pile up exactly where the work is hardest.

Counting a failure as wrong on every field is honest because the pipeline delivered nothing usable for that company, and a downstream user still needs the numbers. Dropping failures would score the model only on the easy cases and inflate accuracy.

> ### [OPTIONAL] A2. Run your own model
>
> If you have an LLM available, set `BACKEND` and `MODEL` above, run your model on **E03, E09, E19** with `temperature=0`, parse the outputs with `parse_response` and compare them with the recorded greedy answers and with `gold`. Did your model make the same mistakes?

In [6]:
# Skipped (optional): no API key / local model available for this run.
# To run it: set BACKEND and MODEL in the setup cell, then call
#   ask_llm(PROMPT.format(text=...), temperature=0) and parse_response(...) on E03, E09, E19.

---
# PART B. Measure the instrument

## B1. Accuracy per field

**Goal.** Create

* `field_correct(field, pred_value, gold_value)`: True if both are missing; False if only one is missing; for `company` a case- and punctuation-insensitive containment match; for years an exact match; for numbers a **relative tolerance of 2%**;
* `score(df)`: one row per (excerpt, field) with `id`, `field`, `pred`, `gold`, `correct`. Excerpts whose response failed to parse must appear with `correct = False` on every field;
* `greedy_scores = score(...)` for the **greedy** (temperature 0) answers, and a table of accuracy per field.

> **Hint.** Missing values come back as `None` or `NaN`: test with `pd.isna`. To include the failures, reindex the greedy predictions on `gold.index`.

In [7]:
def norm_name(s):
    return re.sub(r"[^a-z0-9 ]", "", str(s).lower()).strip()


def field_correct(field, pred_value, gold_value):
    p_na, g_na = pd.isna(pred_value), pd.isna(gold_value)
    if p_na and g_na:
        return True
    if p_na or g_na:
        return False
    if field == "company":
        p, g = norm_name(pred_value), norm_name(gold_value)
        return (g in p) or (p in g)
    if field in ("report_year", "net_zero_year"):
        return int(pred_value) == int(gold_value)
    return abs(float(pred_value) - float(gold_value)) <= 0.02 * abs(float(gold_value))   # 2% relative


def score(df):
    """df: one row per excerpt, indexed by id (missing ids = failed parse). One row per (excerpt, field)."""
    df = df.reindex(gold.index)
    out = []
    for i in gold.index:
        for f in FIELDS:
            p = df.loc[i, f] if f in df.columns else np.nan
            g = gold.loc[i, f] if f != "company" else gold.loc[i, "company"]
            out.append({"id": i, "field": f, "pred": p, "gold": g, "correct": field_correct(f, p, g)})
    return pd.DataFrame(out)


greedy = pred[pred.kind == "greedy"].set_index("id")
greedy_scores = score(greedy)

acc_field = greedy_scores.groupby("field", sort=False).correct.mean().rename("accuracy").to_frame()
acc_field["n_correct"] = greedy_scores.groupby("field", sort=False).correct.sum()
print("Greedy (T=0) accuracy per field")
display(acc_field)
print(f"Overall: {greedy_scores.correct.mean():.3f}  ({greedy_scores.correct.sum()} / {len(greedy_scores)})")

Greedy (T=0) accuracy per field


,accuracy,n_correct
field,,
company,0.950,19
report_year,0.900,18
scope1_tco2,0.750,15
scope2_tco2,0.850,17
production_mt,0.900,18
net_zero_year,0.850,17


Overall: 0.867  (104 / 120)


### Q3 [QUANT + QUAL] Where and how does it fail?

(a) Give the overall accuracy and the **worst field**. (b) Print every wrong (excerpt, field) with `pred` and `gold`, read the excerpt, and sort the errors into **error types** (e.g. unit conversion, wrong year in a table, wrong scope, a value invented for a missing field, parse failure). Which type is most frequent? (c) Which of these errors would **you** also have been likely to make in Q1?

In [8]:
wrong = greedy_scores[~greedy_scores.correct].copy()
display(wrong.reset_index(drop=True))

print("\nExcerpts behind the errors:")
for i in wrong.id.unique():
    print(f"\n{i}: {excerpts.set_index('id').text[i]}")

,id,field,pred,gold,correct
0,E03,scope1_tco2,"4,108,000.000","41,080,000.000",False
1,E03,scope2_tco2,"512,000.000","5,120,000.000",False
2,E05,scope1_tco2,"42,042,000.000",NaN,False
3,E07,scope1_tco2,"14,200,000.000","13,800,000.000",False
4,E08,report_year,"2,025.000",2024,False
5,E09,scope1_tco2,"8,700,000.000",NaN,False
6,E10,net_zero_year,"2,030.000",<NA>,False
7,E11,company,NaN,Anatolia Celik,False
8,E11,report_year,NaN,2024,False
9,E11,scope1_tco2,NaN,"4,120,000.000",False



Excerpts behind the errors:

E03: Lianhe Iron & Steel Group - Environmental, Social and Governance Report 2023. In 2023 the Group produced 21.4 million tonnes of crude steel. Greenhouse gas emissions (unit: 10,000 tonnes CO2): direct emissions (Scope 1) 4,108; indirect emissions from purchased electricity (Scope 2) 512. The Group will strive to peak carbon emissions before 2030 and achieve carbon neutrality by 2060.

E05: Tamarind Steel Ltd - Business Responsibility and Sustainability Report FY2024. Crude steel production rose 6% to 18.2 million tonnes. Our CO2 emission intensity improved to 2.31 tCO2 per tonne of crude steel (tcs), from 2.38 tCO2/tcs a year earlier. Tamarind Steel targets net zero by 2070, in line with India's national commitment.

E07: Great Lakes Ironworks Inc. - 2024 Sustainability Report. Key performance data: Scope 1 emissions (MtCO2e) 2022: 14.9 | 2023: 14.2 | 2024: 13.8. Scope 2 emissions (MtCO2e) 2022: 1.1 | 2023: 1.0 | 2024: 0.96. Raw steel production (Mt) 2

**Your answer (Q3):**

**(a)** Overall greedy accuracy is **104/120 = 86.7%**. The worst field is **`scope1_tco2` (75%, 15/20)**.

**(b)** The 16 wrong (excerpt, field) pairs fall into these types:
- **Value invented or assigned to a field with no figure (5):** E05 (computed 2.31 x 18.2 = 42 Mt although only an intensity is reported), E09 (combined Scope 1+2 put into Scope 1), E18 (Scope 2 = 0 instead of null), E10 (2030 intensity target taken as net zero), E20 (2026 "first coil" date taken as net zero).
- **Unit conversion (3):** E03 Scope 1 and Scope 2 (10,000-tonne table unit ignored), E19 production (short tons not converted).
- **Wrong year or period (2):** E07 Scope 1 taken from the 2023 column (14.2 instead of 13.8), E08 report year 2025 (publication year) instead of 2024 (fiscal year start).
- **Parse failure (6 fields, 1 response):** E11, a string in a numeric field.

The most frequent *incident* is the invented value (5); the parse failure is the biggest by field count, because one failed response wipes out six fields.

**(c)** I would be most likely to repeat the unit errors (E03's x10,000 and E19's short tons) and, when rushed, the temptation to "compute" a missing Scope 1 (E05). Reading the wrong column in the E07 table is also plausible when tired. The parse failure is a pure machine error.

## B2. An interval on accuracy

"91% accurate" on 120 fields is an **estimate** from a small sample, just like the utilisation rate in Lab 3. Two ways to put a range on it:

1. **Bayesian (Beta-Binomial).** With a uniform prior Beta(1, 1) and *k* correct out of *n*, the posterior is **Beta(1 + k, 1 + n - k)**. Its 5th and 95th percentiles give a 90% credible interval.
2. **Bootstrap over excerpts.** Resample **excerpts** (not fields) with replacement, recompute accuracy, repeat 2,000 times, take the 5th and 95th percentiles.

**Goal.** Create `k`, `n`, `beta_ci` and `boot_ci` (both 90%).

> **Hint.** `stats.beta(a, b).ppf([0.05, 0.95])`. For the bootstrap, first compute per excerpt the number correct and the number of fields (`groupby("id").correct.agg(["sum", "count"])`), then use the same `rng.integers(...)` trick as Lab 3.

In [9]:
k = int(greedy_scores.correct.sum())
n = int(len(greedy_scores))
beta_ci = stats.beta(1 + k, 1 + n - k).ppf([0.05, 0.95])

# bootstrap over EXCERPTS (the six fields of one excerpt move together)
per_ex = greedy_scores.groupby("id").correct.agg(["sum", "count"])
rng = np.random.default_rng(SEED)
B = 2000
idx = rng.integers(0, len(per_ex), size=(B, len(per_ex)))
boot_acc = per_ex["sum"].values[idx].sum(1) / per_ex["count"].values[idx].sum(1)
boot_ci = np.percentile(boot_acc, [5, 95])

print(f"accuracy = {k}/{n} = {k/n:.3f}")
print(f"Beta-Binomial 90% interval : [{beta_ci[0]:.3f}, {beta_ci[1]:.3f}]  width {np.diff(beta_ci)[0]:.3f}")
print(f"Bootstrap     90% interval : [{boot_ci[0]:.3f}, {boot_ci[1]:.3f}]  width {np.diff(boot_ci)[0]:.3f}")
print(f"P(accuracy > 0.95 | data) under the Beta posterior: {1 - stats.beta(1 + k, 1 + n - k).cdf(0.95):.4f}")
print("Errors per excerpt:\n", (per_ex['count'] - per_ex['sum']).loc[lambda s: s > 0].to_dict())

accuracy = 104/120 = 0.867
Beta-Binomial 90% interval : [0.806, 0.908]  width 0.102
Bootstrap     90% interval : [0.775, 0.933]  width 0.158
P(accuracy > 0.95 | data) under the Beta posterior: 0.0001
Errors per excerpt:
 {'E03': 2, 'E05': 1, 'E07': 1, 'E08': 1, 'E09': 1, 'E10': 1, 'E11': 6, 'E18': 1, 'E19': 1, 'E20': 1}


### Q4 [QUANT + QUAL] How sure are we about "accurate"?

(a) Report both intervals. Which is wider, and why? (Hint: are the six fields of one excerpt independent draws? Think of E11.) (b) Roughly how many excerpts would you need to label to **halve** the width of the interval? (c) A vendor advertises "95% extraction accuracy". Is your result consistent with that claim?

**Your answer (Q4):**

**(a)** Beta-Binomial 90% interval: **[0.806, 0.908]** (width 0.102). Bootstrap over excerpts: **[0.775, 0.933]** (width 0.158). The bootstrap is wider because the six fields of an excerpt are not independent draws: 6 of the 16 errors come from E11 alone. The Beta-Binomial treats 120 independent trials, but the effective sample is closer to 20 excerpts, so it is over-confident.

**(b)** The width shrinks like 1/sqrt(n), so halving it needs about **4x as many excerpts**: roughly 80 instead of 20 (about 480 fields).

**(c)** No. 95% lies above the upper end of both intervals (0.908 and 0.933), and the Beta posterior gives P(accuracy > 0.95) of about 0.0001. Caveat: this is a small, fictional, deliberately tricky sample, so it says little about the vendor's own benchmark.

## B3. What do the errors do downstream?

A lender wants the **sector total of Scope 1 emissions** and the **median emission intensity** (tCO2 per tonne of steel) of these 20 companies.

**Goal.** Create a DataFrame `downstream` indexed by excerpt with `scope1_llm`, `scope1_gold`, `error_t` (= LLM - gold, in tCO2, treating a missing value as 0 for the total) and print: total Scope 1 according to the LLM and to gold (MtCO2), the **net** and the **absolute** error, and the median intensity (Scope 1 / production, only where both are present) for LLM and gold.

In [10]:
dn = pd.DataFrame({
    "scope1_llm":  greedy.reindex(gold.index).scope1_tco2,
    "scope1_gold": gold.scope1_tco2.astype(float),
    "prod_llm":    greedy.reindex(gold.index).production_mt,
    "prod_gold":   gold.production_mt,
})
downstream = dn[["scope1_llm", "scope1_gold"]].copy()
downstream["error_t"] = dn.scope1_llm.fillna(0) - dn.scope1_gold.fillna(0)   # missing counted as 0 for the total

tot_llm  = dn.scope1_llm.fillna(0).sum() / 1e6
tot_gold = dn.scope1_gold.fillna(0).sum() / 1e6
net_err  = downstream.error_t.sum() / 1e6
abs_err  = downstream.error_t.abs().sum() / 1e6
print(f"Total Scope 1  LLM : {tot_llm:8.2f} MtCO2")
print(f"Total Scope 1  gold: {tot_gold:8.2f} MtCO2")
print(f"Net error     : {net_err:+8.2f} MtCO2  ({net_err/tot_gold:+.1%})")
print(f"Absolute error: {abs_err:8.2f} MtCO2  ({abs_err/tot_gold:.1%})")

# intensity = Scope 1 / production (tCO2 per t steel), only where both present
inten_llm  = (dn.scope1_llm  / (dn.prod_llm  * 1e6)).dropna()
inten_gold = (dn.scope1_gold / (dn.prod_gold * 1e6)).dropna()
print(f"\nMedian intensity  LLM : {inten_llm.median():.3f} tCO2/t   (n={len(inten_llm)})")
print(f"Median intensity  gold: {inten_gold.median():.3f} tCO2/t   (n={len(inten_gold)})")

print("\nLargest contributors to the total error:")
display(downstream.assign(abs_error_Mt=downstream.error_t.abs() / 1e6)
        .sort_values("abs_error_Mt", ascending=False).head(5))

Total Scope 1  LLM :   252.11 MtCO2
Total Scope 1  gold:   242.06 MtCO2
Net error     :   +10.05 MtCO2  (+4.2%)
Absolute error:    92.23 MtCO2  (38.1%)

Median intensity  LLM : 1.895 tCO2/t   (n=18)
Median intensity  gold: 1.865 tCO2/t   (n=17)

Largest contributors to the total error:


,scope1_llm,scope1_gold,error_t,abs_error_Mt
id,,,,
E05,"42,042,000.000",NaN,"42,042,000.000",42.042
E03,"4,108,000.000","41,080,000.000","-36,972,000.000",36.972
E09,"8,700,000.000",NaN,"8,700,000.000",8.700
E11,NaN,"4,120,000.000","-4,120,000.000",4.120
E07,"14,200,000.000","13,800,000.000","400,000.000",0.400


### Q5 [QUANT + QUAL] Accuracy is a mean, the damage is in the tail

(a) Report the net and absolute error on the sector total. Which **two** excerpts contribute most? (b) Why can a 90% field accuracy hide a large error on a portfolio total, while the **median** intensity barely moves? Connect to Session 4 ("the mean hides the tail"). (c) What would you report to the lender alongside the total?

**Your answer (Q5):**

**(a)** The LLM total is 252.11 MtCO2 against 242.06 for gold: a **net error of +10.05 Mt (+4.2%)** but an **absolute error of 92.23 Mt (38.1%)**. The two biggest contributors are **E05** (+42.0 Mt, a Scope 1 computed from an intensity when none is reported) and **E03** (-37.0 Mt, the 10,000-tonne unit missed); E09 (+8.7 Mt) comes next.

**(b)** Accuracy counts every field as 0 or 1 whatever its size, but in a sum one field that is off by a factor of 10 or invented from nothing is worth tens of megatonnes. Here the errors of opposite sign (E05 up, E03 down) nearly cancel in the net figure, which hides how wrong the individual rows are. The median intensity barely moves (1.895 vs 1.865 tCO2/t) because a median ignores the extremes. This is "the mean hides the tail" in reverse: the sum is driven by the tail, and an accuracy average hides it.

**(c)** Report the total as a range with the absolute error, not just the net, and say how many companies are flagged or hand-checked. List the biggest contributors for human verification and give the median intensity next to the total as the more robust figure.

---
# PART C. Self-consistency: ask five times

At temperature 0.7 the model **samples** its answer, so asking the same question five times can give different answers. Idea: use the **majority vote** as the answer and the **agreement** (share of samples that agree with the vote) as an uncertainty signal. This is an **ensemble**, like the multi-model comparison of Session 4.

## C1. Vote and agreement

**Goal.** Create `votes`: one row per (excerpt, field) with `vote` (most frequent answer among the 5 samples, missing counted as an answer), `agreement` (count of the vote / 5) and `correct`. Compare the vote's accuracy with the greedy accuracy, and show the accuracy for agreement **5/5**, **4/5** and **3/5 or less**.

> **Hint.** To count values that may be `None`, `NaN` or numbers, turn each into a string key first: `json.dumps(None if pd.isna(v) else v)`.

In [11]:
def key(v):
    if v is None or (not isinstance(v, str) and pd.isna(v)):
        return json.dumps(None)
    if isinstance(v, str):
        return json.dumps(norm_name(v))      # company: ignore case/punctuation (S.A. etc. handled by containment below)
    return json.dumps(float(v))


samples = pred[pred.kind == "sample"]
vrows = []
for i in gold.index:
    s = samples[samples.id == i]
    for f in FIELDS:
        keys = [key(v) for v in s[f]]
        vote_key, cnt = pd.Series(keys).value_counts().iloc[0:1].reset_index().values[0] if keys else (json.dumps(None), 0)
        vote = json.loads(vote_key)
        if f == "company" and vote is not None:      # recover a display name from the first sample that voted it
            vote = s[f][[key(v) == vote_key for v in s[f]]].iloc[0]
        vrows.append({"id": i, "field": f, "vote": vote, "agreement": cnt / 5,
                      "correct": field_correct(f, vote, gold.loc[i, f])})
votes = pd.DataFrame(vrows)

print(f"Greedy accuracy : {greedy_scores.correct.mean():.3f}")
print(f"Vote accuracy   : {votes.correct.mean():.3f}")
kv, nv = int(votes.correct.sum()), len(votes)
vote_ci = stats.beta(1 + kv, 1 + nv - kv).ppf([0.05, 0.95])
print(f"Vote accuracy {kv}/{nv}: 90% Beta interval [{vote_ci[0]:.3f}, {vote_ci[1]:.3f}]")
votes["bin"] = pd.cut(votes.agreement, [0, .6, .8, 1.0], labels=["<=3/5", "4/5", "5/5"])
bins = votes.groupby("bin", observed=True).correct.agg(["mean", "sum", "count"]).loc[["5/5", "4/5", "<=3/5"]]
bins.columns = ["accuracy", "n_correct", "n_fields"]
display(bins)

Greedy accuracy : 0.867
Vote accuracy   : 0.950
Vote accuracy 114/120: 90% Beta interval [0.904, 0.973]


,accuracy,n_correct,n_fields
bin,,,
5/5,0.978,91,93
4/5,0.933,14,15
<=3/5,0.750,9,12


### Q6 [QUANT + QUAL] Is disagreement a useful signal?

(a) Report the greedy and vote accuracies and the accuracy in each agreement bin. (b) Is low agreement a good warning sign? (c) Compute the accuracy for 5/5 with a Beta(1,1) 90% interval: can you claim it is above 95%?

In [12]:
b55 = votes[votes.agreement == 1.0]
k5, n5 = int(b55.correct.sum()), len(b55)
ci5 = stats.beta(1 + k5, 1 + n5 - k5).ppf([0.05, 0.95])
p_above = 1 - stats.beta(1 + k5, 1 + n5 - k5).cdf(0.95)
print(f"5/5 bin: {k5}/{n5} = {k5/n5:.3f},  90% interval [{ci5[0]:.3f}, {ci5[1]:.3f}],  P(acc > 0.95) = {p_above:.2f}")

lo = votes[votes.agreement < 1.0]
print(f"agreement < 5/5: {int(lo.correct.sum())}/{len(lo)} correct ({lo.correct.mean():.3f}); "
      f"share of all wrong votes caught by low agreement: {(~lo.correct).sum()}/{(~votes.correct).sum()}")

5/5 bin: 91/93 = 0.978,  90% interval [0.935, 0.991],  P(acc > 0.95) = 0.85
agreement < 5/5: 23/27 correct (0.852); share of all wrong votes caught by low agreement: 4/6


**Your answer (Q6):**

**(a)** Greedy accuracy is **86.7%**; majority-vote accuracy is **95.0%** (114/120, 90% interval [0.904, 0.973]). The vote also repairs the E11 parse failure, because it uses the five sampled responses. By agreement: **5/5: 97.8%** (91/93), **4/5: 93.3%** (14/15), **3/5 or less: 75.0%** (9/12).

**(b)** Yes, moderately. Fields below 5/5 are 85.2% correct versus 97.8% at 5/5, and they hold 4 of the 6 wrong votes in only 27 of 120 fields. The 4/5 bin is only slightly worse than 5/5 and has few fields, so the clear warning sign is agreement of 3/5 or less.

**(c)** 5/5: 91/93 = 97.8%, 90% interval **[0.935, 0.991]**, and P(accuracy > 0.95) = 0.85. The interval includes values below 95%, so I cannot claim it with confidence. In IPCC words it is only *likely* (66-100%) that it exceeds 95%, not *very likely* (90-100%).

## C2. Confidently wrong

### Q7 [QUAL] The errors that resampling cannot see

Print the (excerpt, field) pairs where **all five samples agree** and the vote is **wrong**. Read the excerpts. (a) What do these errors have in common? (b) Using Session 4 vocabulary, why does asking the same model more times not reveal them? Which fix would (a different prompt, a second model, a unit-conversion rule, a human)? (c) Name the analogue in the Session 4 worked example.

In [13]:
cw = votes[(votes.agreement == 1.0) & (~votes.correct)].copy()
cw["gold"] = [gold.loc[r.id, r.field] for r in cw.itertuples()]
display(cw[["id", "field", "vote", "gold", "agreement"]].reset_index(drop=True))
for i in cw.id.unique():
    print(f"\n{i}: {excerpts.set_index('id').text[i]}")

,id,field,vote,gold,agreement
0,E03,scope1_tco2,"4,108,000.000","41,080,000.000",1.000
1,E19,production_mt,3.300,2.990,1.000



E03: Lianhe Iron & Steel Group - Environmental, Social and Governance Report 2023. In 2023 the Group produced 21.4 million tonnes of crude steel. Greenhouse gas emissions (unit: 10,000 tonnes CO2): direct emissions (Scope 1) 4,108; indirect emissions from purchased electricity (Scope 2) 512. The Group will strive to peak carbon emissions before 2030 and achieve carbon neutrality by 2060.

E19: Bluewater Iron Corp. - 2024 Environmental Report. Shipments and production: we produced 3.3 million short tons of raw steel at our DRI-EAF facilities. Scope 1 emissions: 5.1 million metric tonnes CO2e; Scope 2: 410,000 metric tonnes CO2e. Bluewater targets net zero by 2050.


**Your answer (Q7):**

All five samples agree and the vote is wrong for two fields: **E03 Scope 1** (4,108,000 instead of 41,080,000) and **E19 production** (3.3 instead of 2.99 Mt).

**(a)** Both are **unit-conversion errors**: a table unit ("10,000 tonnes") and a different ton ("short tons"). The model misreads the same way each time.

**(b)** Resampling averages out *random* variation (aleatoric-like noise in the sampling), but these errors are *systematic* (epistemic): they come from the model's fixed misunderstanding, so every sample repeats it. Same model plus same prompt means five correlated draws, not five independent opinions. What would catch them: a **deterministic unit rule** for E19 (detect "short ton" and convert), a **plausibility check** for E03 (implied intensity 4.1/21.4 = 0.19 tCO2/t is far below the 1-3 typical for steel), a **second, different model**, or a **human** for flagged unit keywords.

**(c)** It is the straight-line model in the Session 4 worked example: its interval was narrow and still missed 3 of 5 outcomes, because the true process was exponential. Intervals only quantify uncertainty *inside* the chosen model, and the fix was to compare a different model (the ensemble). Our five samples are repeated draws from one model, like refitting the same line on resampled data.

## C3. The model's own confidence

Each response contains a `confidence` the model wrote about itself.

### Q8 [QUANT + QUAL] Is stated confidence calibrated?

For the **sample** responses, compute the mean stated confidence and the share of responses with **all six fields correct**. Then group responses by confidence (`pd.cut(..., [0, .90, .93, .96, 1])`) and show the share fully correct in each group. Is the model **over-confident**? Is its confidence still useful for **ranking** responses?

In [14]:
srows = []
for r in samples.itertuples():
    ok = all(field_correct(f, getattr(r, f), gold.loc[r.id, f]) for f in FIELDS)
    srows.append({"id": r.id, "sample": r.sample, "confidence": r.confidence, "all_correct": ok})
sc = pd.DataFrame(srows)

print(f"responses: {len(sc)}   (parse failures excluded: {sum(f['kind']=='sample' for f in failures)})")
print(f"mean stated confidence : {sc.confidence.mean():.3f}")
print(f"share fully correct    : {sc.all_correct.mean():.3f}")

sc["conf_bin"] = pd.cut(sc.confidence, [0, .90, .93, .96, 1])
calib = sc.groupby("conf_bin", observed=True).agg(n=("all_correct", "size"), mean_conf=("confidence", "mean"),
                                                  share_correct=("all_correct", "mean"))
display(calib)

# ranking quality: AUC that a fully-correct response has a higher confidence than a wrong one
pos, neg = sc[sc.all_correct].confidence, sc[~sc.all_correct].confidence
auc = (pos.values[:, None] > neg.values[None, :]).mean() + 0.5 * (pos.values[:, None] == neg.values[None, :]).mean()
print(f"AUC of stated confidence for 'fully correct': {auc:.2f}  (0.5 = no information)")

responses: 100   (parse failures excluded: 0)
mean stated confidence : 0.928
share fully correct    : 0.550


,n,mean_conf,share_correct
conf_bin,,,
"(0.0, 0.9]",21,0.890,0.286
"(0.9, 0.93]",35,0.921,0.457
"(0.93, 0.96]",38,0.947,0.737
"(0.96, 1.0]",6,0.978,0.833


AUC of stated confidence for 'fully correct': 0.73  (0.5 = no information)


**Your answer (Q8):**

Over the 100 sample responses the **mean stated confidence is 0.928** but only **55.0%** have all six fields correct, so the model is clearly **over-confident**. (This is a strict test: it needs all six fields right, but even so the gap is large.) By confidence bin the share fully correct rises steadily: (0, 0.90] 28.6%, (0.90, 0.93] 45.7%, (0.93, 0.96] 73.7%, (0.96, 1] 83.3% (only 6 responses in the top bin). So the stated confidence is **not calibrated but still useful for ranking**: AUC is 0.73, well above 0.5. It can order responses for review, but its value must not be read as a probability.

## C4. A review policy with a price tag

A human analyst can check a field for **EUR 4**. An error that reaches the lender's report costs on average **EUR 150** (restatement, analyst time, reputational risk). Three policies:

| Policy | Rule |
|---|---|
| **P0** | trust the majority vote everywhere |
| **P1** | send to a human every field with agreement **< 5/5** |
| **P2** | send to a human every field with agreement **<= 3/5** |

Assume the human fixes every field they review.

### Q9 [QUANT + QUAL] Which policy?

For each policy compute the number of fields reviewed, the number of errors that remain, and the total cost. Which policy is cheapest? Which errors survive even the best policy, and what one extra check would catch them?

In [15]:
REVIEW_COST, ERROR_COST = 4, 150
policies = {"P0 trust vote": votes.agreement < 0,          # nobody reviewed
            "P1 review < 5/5": votes.agreement < 1.0,
            "P2 review <= 3/5": votes.agreement <= 0.6}
res = []
for name, reviewed in policies.items():
    remaining = (~votes.correct & ~reviewed).sum()          # human fixes everything they review
    res.append({"policy": name, "fields_reviewed": int(reviewed.sum()), "errors_remaining": int(remaining),
                "review_cost": REVIEW_COST * int(reviewed.sum()), "error_cost": ERROR_COST * int(remaining),
                "total_cost_EUR": REVIEW_COST * int(reviewed.sum()) + ERROR_COST * int(remaining)})
policy_table = pd.DataFrame(res).set_index("policy")
display(policy_table)
print("For reference, reviewing all 120 fields would cost EUR", REVIEW_COST * len(votes))

best = policy_table.total_cost_EUR.idxmin()
print("Cheapest:", best)
surv = votes[(~votes.correct) & ~policies[best]].copy()
surv["gold"] = [gold.loc[r.id, r.field] for r in surv.itertuples()]
print("Errors surviving the best policy:"); display(surv[["id", "field", "vote", "gold", "agreement"]].reset_index(drop=True))

,fields_reviewed,errors_remaining,review_cost,error_cost,total_cost_EUR
policy,,,,,
P0 trust vote,0,6,0,900,900
P1 review < 5/5,27,2,108,300,408
P2 review <= 3/5,12,3,48,450,498


For reference, reviewing all 120 fields would cost EUR 480
Cheapest: P1 review < 5/5
Errors surviving the best policy:


,id,field,vote,gold,agreement
0,E03,scope1_tco2,"4,108,000.000","41,080,000.000",1.000
1,E19,production_mt,3.300,2.990,1.000


**Your answer (Q9):**

| Policy | Fields reviewed | Errors left | Total cost |
|---|---|---|---|
| P0 trust the vote | 0 | 6 | EUR 900 |
| P1 review < 5/5 | 27 | 2 | **EUR 408** |
| P2 review <= 3/5 | 12 | 3 | EUR 498 |

**P1 is the cheapest**, also below checking all 120 fields by hand (EUR 480). The errors that survive it are the two confidently wrong unit errors, E03 Scope 1 and E19 production, both with agreement 5/5. One extra check would catch them: an automatic unit and plausibility rule (flag "short tons" or "10,000 tonnes" keywords, and check that Scope 1 / production falls in a plausible intensity range), with flagged fields going to a human.

---
# PART D. Retrieval: finding the right paragraph

Real reports are 100+ pages, longer than you want to send to a model. **RAG** first **retrieves** the most relevant chunks, then asks the LLM to answer from them. If retrieval misses, the LLM never sees the answer.

`corpus["chunks"]` is a (fictional) report cut into 16 chunks; `corpus["queries"]` are 6 questions with the index of the chunk that answers each one.

## D1. TF-IDF retrieval

**Goal.** Represent chunks and queries as **TF-IDF vectors** (`TfidfVectorizer(stop_words="english")`), score each chunk by **cosine similarity** with the query, and build a table `retrieval` with, per query, the top-3 chunk indices, whether the relevant chunk is **rank 1** (`hit@1`) and whether it is in the **top 3** (`hit@3`).

> **Hint.** `from sklearn.feature_extraction.text import TfidfVectorizer`, `from sklearn.metrics.pairwise import cosine_similarity`; `np.argsort(-scores)[:3]`.

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

chunks, queries = corpus["chunks"], corpus["queries"]
chunk_texts = [c["text"] if isinstance(c, dict) else c for c in chunks]

vec = TfidfVectorizer(stop_words="english")
X = vec.fit_transform(chunk_texts)
Q = vec.transform([q["query"] for q in queries])
sims = cosine_similarity(Q, X)

rrows = []
for qi, q in enumerate(queries):
    top3 = np.argsort(-sims[qi])[:3].tolist()
    rel = q["relevant"]
    rrows.append({"qid": q["qid"], "query": q["query"], "relevant": rel[0], "top3": top3,
                  "hit@1": top3[0] in rel, "hit@3": any(t in rel for t in top3)})
retrieval = pd.DataFrame(rrows)
display(retrieval)
print(f"hit@1 = {retrieval['hit@1'].mean():.2f}   hit@3 = {retrieval['hit@3'].mean():.2f}")

,qid,query,relevant,top3,hit@1,hit@3
0,Q1,What were Scope 1 emissions in 2024?,3,"[1, 5, 12]",False,False
1,Q2,net zero target year,5,"[5, 0, 13]",True,True
2,Q3,crude steel production 2024,2,"[2, 3, 0]",True,True
3,Q4,investment in hydrogen DRI capex,6,"[6, 0, 2]",True,True
4,Q5,impact of the carbon price on EBITDA,8,"[8, 13, 7]",True,True
5,Q6,flooding exposure of our plants,9,"[3, 6, 1]",False,False


hit@1 = 0.67   hit@3 = 0.67


### Q10 [QUANT + QUAL] Why does retrieval miss?

(a) Report hit@1 and hit@3. (b) For each failed query, print the query, the chunk it retrieved first and the relevant chunk. Why did TF-IDF fail? (c) **Embeddings** (dense vectors from a neural model) compare meaning rather than words. Which failure would they most likely fix? (d) In a RAG pipeline, why is a retrieval miss more dangerous than an extraction error?

In [17]:
for qi, row in retrieval[~retrieval["hit@1"]].iterrows():
    top1 = row.top3[0]
    print(f"{row.qid}: {row['query']!r}   (relevant chunk {row.relevant} has similarity "
          f"{sims[qi][row.relevant]:.3f}, retrieved chunk {top1} has {sims[qi][top1]:.3f})")
    print(f"  retrieved #1 (chunk {top1}, sim {sims[qi][top1]:.3f}): {chunk_texts[top1]}")
    print(f"  relevant     (chunk {row.relevant}, sim {sims[qi][row.relevant]:.3f}): {chunk_texts[row.relevant]}")
    print(f"  shared vocabulary with the relevant chunk: "
          f"{sorted(set(vec.build_analyzer()(row['query'])) & set(vec.build_analyzer()(chunk_texts[row.relevant])))}\n")

Q1: 'What were Scope 1 emissions in 2024?'   (relevant chunk 3 has similarity 0.184, retrieved chunk 1 has 0.338)
  retrieved #1 (chunk 1, sim 0.338): About this report. Greenhouse gas reporting follows the GHG Protocol. Scope 1 covers emissions from sources we own or control, Scope 2 covers purchased electricity and heat, and Scope 3 covers our value chain.
  relevant     (chunk 3, sim 0.184): Climate performance. Direct greenhouse gas emissions from our blast furnaces, coke ovens and sinter plants totalled 24.19 MtCO2e in 2024, an intensity of 1.92 tCO2e per tonne of crude steel.
  shared vocabulary with the relevant chunk: ['2024', 'emissions']

Q6: 'flooding exposure of our plants'   (relevant chunk 9 has similarity 0.000, retrieved chunk 3 has 0.208)
  retrieved #1 (chunk 3, sim 0.208): Climate performance. Direct greenhouse gas emissions from our blast furnaces, coke ovens and sinter plants totalled 24.19 MtCO2e in 2024, an intensity of 1.92 tCO2e per tonne of crude steel.
  rele

**Your answer (Q10):**

**(a)** hit@1 = **4/6 = 0.67** and hit@3 = **0.67**, so the top 3 does not help: the two misses are not close.

**(b)** **Q1** ("What were Scope 1 emissions in 2024?"): TF-IDF returned chunk 1, the definition of Scope 1/2/3, which repeats the words "Scope 1" and "emissions". The right chunk 3 says "Direct greenhouse gas emissions ... 24.19 MtCO2e" and never uses the term "Scope 1" (similarity 0.18 vs 0.34). **Q6** ("flooding exposure of our plants"): the relevant chunk 9 talks about "riverine inundation" and "exposed", and shares *no* word with the query, so its similarity is 0. TF-IDF matches words, not meaning, and fails on synonyms.

**(c)** Embeddings would most likely fix **Q6** (flooding and inundation are close in meaning) and probably **Q1** (Scope 1 is close to direct emissions).

**(d)** A retrieval miss is silent. The LLM never sees the answer and either says nothing or answers fluently from the wrong chunk (for Q1, a definition or an invented number), with no sign that anything went wrong. An extraction error at least sits in the right paragraph, where a reviewer can check it against the source. Our Part B and C measurements also do not count retrieval misses.

> ### [OPTIONAL] D2. Dense embeddings
>
> `pip install sentence-transformers`, then `SentenceTransformer("all-MiniLM-L6-v2").encode(...)` to embed chunks and queries, and redo D1 with cosine similarity on the embeddings. Does hit@1 improve? Which query still fails?

In [18]:
# Skipped (optional): needs `sentence-transformers` and a model download.
# from sentence_transformers import SentenceTransformer
# m = SentenceTransformer("all-MiniLM-L6-v2"); E = m.encode(chunk_texts); Qe = m.encode([q["query"] for q in queries])

---
# PART E. Communicate

### Q11 [QUAL] Should the bank adopt this pipeline?

Write a note of **about 80 words** to the bank's **Head of ESG Data**, who must decide whether to use this LLM pipeline to build a database of steel-company emissions. Include: the **accuracy with its interval**, the **main failure mode**, your **recommended review policy** and its cost, and **one limitation** of this evaluation. Plain language, no jargon. Use a calibrated word from the IPCC scale at least once.

**Your answer (Q11):**

> Dear Head of ESG Data, we tested the pipeline on 20 reports. Asking the model five times and taking the majority gets about 95% of fields right (90% interval: 90-97%). Its main weakness is unit mistakes, like "10,000 tonnes" or short tons, which repeat on every retry. We recommend human review of every field where the five answers disagree plus an automatic unit check; that cost about EUR 408 per 120 fields. Limitation: only 20 fictional reports. Real reports are likely harder.